# Crop Yield Prediction Analysis

This notebook demonstrates the process of analyzing agricultural data and building machine learning models for crop yield prediction. We'll use Random Forest and Support Vector Regression models to predict crop yields based on various environmental factors.

## 1. Import Required Libraries

First, let's import all the necessary libraries for data analysis and machine learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Set style for visualizations
plt.style.use('seaborn')
sns.set_palette('husl')

# Display all columns in pandas dataframes
pd.set_option('display.max_columns', None)

## 2. Load and Explore Data

Let's load our agricultural dataset and perform initial exploratory data analysis.

In [ ]:
# Load the dataset
# Note: Replace 'data/crop_data.csv' with your actual dataset path
df = pd.read_csv('../data/crop_data.csv')

# Display basic information about the dataset
print("Dataset Info:")
print("-" * 50)
print(df.info())
print("\nFirst few rows:")
print("-" * 50)
display(df.head())

# Display summary statistics
print("\nSummary Statistics:")
print("-" * 50)
display(df.describe())

## 3. Data Preprocessing

Now let's prepare our data for modeling by handling missing values, scaling features, and splitting into training and testing sets.

In [ ]:
# Handle missing values
df = df.fillna(df.mean(numeric_only=True))

# Separate features and target
X = df.drop('yield', axis=1)  # Adjust 'yield' to match your target column name
y = df['yield']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Testing set shape:", X_test_scaled.shape)

## 4. Model Training and Evaluation

Let's train both Random Forest and Support Vector Regression models and compare their performance.

In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_predictions = rf_model.predict(X_test_scaled)

# Train SVR model
svr_model = SVR(kernel='rbf', C=1.0)
svr_model.fit(X_train_scaled, y_train)
svr_predictions = svr_model.predict(X_test_scaled)

# Evaluate models
def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} Performance Metrics:")
    print("-" * 50)
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}\n")

# Print evaluation metrics
evaluate_model(y_test, rf_predictions, "Random Forest")
evaluate_model(y_test, svr_predictions, "Support Vector Regression")

## 5. Visualize Results

Let's create some visualizations to better understand our models' performance.

In [ ]:
# Plot actual vs predicted values for both models
plt.figure(figsize=(15, 5))

# Random Forest
plt.subplot(121)
plt.scatter(y_test, rf_predictions, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.title('Random Forest: Actual vs Predicted')

# SVR
plt.subplot(122)
plt.scatter(y_test, svr_predictions, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.title('SVR: Actual vs Predicted')

plt.tight_layout()
plt.show()

# Plot feature importance for Random Forest
if hasattr(rf_model, 'feature_importances_'):
    plt.figure(figsize=(10, 6))
    feat_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    sns.barplot(data=feat_importance, x='importance', y='feature')
    plt.title('Random Forest Feature Importance')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 6. Save the Best Model

Let's save the best performing model for use in our web application.

In [ ]:
import joblib

# Save the Random Forest model (assuming it performed better)
joblib.dump(rf_model, '../models/crop_yield_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Model and scaler saved successfully!")